# 02 — Inference privacyA query hypervector can be inverted by an attacker to recover the raw image.This notebook zeroes the low-variance dimensions of the query and measures bothsides of the trade: classifier accuracy, and the reconstruction error of anadversarial decoder.Dropping high-variance dimensions is included as a control — it hurts theattacker by a similar amount but destroys accuracy.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import numpy as np
import torch

from dphd import *
from dphd.data import load_dataset, split_to_tensors
from dphd.plotting import plot_sigma_sweep, plot_accuracy_vs_nmse
from dphd.privacy import snr_db

set_seed(21)
device = get_device()
print("device:", device)

## Parameters

In [ ]:
HV_DIM = 1000
SIGMA_B = 0.2        # use sigma_b* from notebook 01
N_EPOCHS = 4
MAX_DROP = 60        # maximum % of dimensions zeroed
DROP_STEP = 5
N_RUNS = 3           # the paper averages over 100
DECODER_EPOCHS = 10
N_SAMPLES = 5000
DOWNSAMPLE = 2

## Data and model

In [ ]:
X, y = load_dataset("mnist", root="../data", n_samples=N_SAMPLES, downsample=DOWNSAMPLE)
fractions = np.arange(0, MAX_DROP + 1, DROP_STEP) / 100.0

from dphd.decoder import train_decoder, nmse

acc_low_runs, acc_high_runs, err_low_runs, err_high_runs = [], [], [], []

for run in range(N_RUNS):
    X_tr, X_te, y_tr, y_te = split_to_tensors(X, y, device=device, seed=42 + run)
    n_classes, n_features = int(torch.unique(y_tr).numel()), X_tr.shape[1]

    basis, bias = make_basis(n_features, HV_DIM, SIGMA_B, device)
    enc_tr, enc_te = encode(X_tr, basis, bias), encode(X_te, basis, bias)

    class_hvs = train_class_hvs(enc_tr, y_tr, n_classes)
    class_hvs = retrain(class_hvs, enc_tr, y_tr, n_epochs=N_EPOCHS)

    print(f"run {run + 1}/{N_RUNS}: training the adversarial decoder ...")
    decoder = train_decoder(enc_tr, X_tr, epochs=DECODER_EPOCHS)

    order = variance_order(enc_te)
    a_lo, a_hi, e_lo, e_hi = [], [], [], []
    for frac in fractions:
        q_lo = drop_dimensions(enc_te, order, frac, mode="low")
        q_hi = drop_dimensions(enc_te, order, frac, mode="high")
        a_lo.append(accuracy(q_lo, class_hvs, y_te) * 100)
        a_hi.append(accuracy(q_hi, class_hvs, y_te) * 100)
        with torch.no_grad():
            e_lo.append(nmse(decoder(q_lo), X_te))
            e_hi.append(nmse(decoder(q_hi), X_te))

    acc_low_runs.append(a_lo); acc_high_runs.append(a_hi)
    err_low_runs.append(e_lo); err_high_runs.append(e_hi)

acc_low = np.mean(acc_low_runs, axis=0)
acc_high = np.mean(acc_high_runs, axis=0)
err_low = np.mean(err_low_runs, axis=0)
err_high = np.mean(err_high_runs, axis=0)
drop_pct = fractions * 100

## Result

In [ ]:
plot_accuracy_vs_nmse(drop_pct, acc_low, acc_high, err_low, err_high)

print(f"dropping {drop_pct[-1]:.0f}% of low-variance dimensions:")
print(f"  accuracy {acc_low[0]:.2f}% -> {acc_low[-1]:.2f}%  ({acc_low[0] - acc_low[-1]:.2f} point drop)")
print(f"  NMSE     {err_low[0]:.3f} -> {err_low[-1]:.3f}  ({(err_low[-1] / err_low[0] - 1) * 100:.0f}% rise)")
print(f"dropping the same share of high-variance dimensions:")
print(f"  accuracy {acc_high[0]:.2f}% -> {acc_high[-1]:.2f}%  ({acc_high[0] - acc_high[-1]:.2f} point drop)")
print(f"  NMSE     {err_high[0]:.3f} -> {err_high[-1]:.3f}")

## Visual checkWhat the attacker actually recovers, before and after the defence.

In [ ]:
import matplotlib.pyplot as plt

side = int(np.sqrt(n_features))
q_clean = drop_dimensions(enc_te, order, 0.0, mode="low")
q_defended = drop_dimensions(enc_te, order, MAX_DROP / 100, mode="low")

with torch.no_grad():
    rec_clean = decoder(q_clean).cpu().numpy()
    rec_defended = decoder(q_defended).cpu().numpy()
original = X_te.cpu().numpy()

fig, axes = plt.subplots(3, 6, figsize=(10, 5.4))
for j in range(6):
    for i, (img, title) in enumerate([
        (original[j], "original"),
        (rec_clean[j], "attacker, no defence"),
        (rec_defended[j], f"attacker, {MAX_DROP}% dropped"),
    ]):
        axes[i, j].imshow(img.reshape(side, side), cmap="gray")
        axes[i, j].axis("off")
        if j == 0:
            axes[i, j].set_title(title, loc="left", fontsize=10)
plt.tight_layout()
plt.show()